SAM 2026

#TP3-4 LSM Tree Store


document modifié le 11/02/2026

**SUJET SEUL, durée 2 séances**

L'objectif est de comprendre, par la pratique, les principes du stockage couramment mis en oeuvre dans les systèmes big data tels que Cassandra et HBase.

# Préparation

Consigne: lire attentivement cette partie et comprendre précisémenet le fonctionnement de l'algorithme de fusion de fichiers: `fusionner_plusieurs_fichiers_csv_tries`

Points importants:
* cet algorithme passe à l'échelle et nécessite **très peu de mémoire** : une seule ligne par fichier doit être en mémoire.
* en cas d'**ex-aequos** sur la clé de tri, l'ordre des fichiers compte et le tri se fait sur l'**indice** du fichier.



In [1]:
import os
import shutil
import random
import numpy as np
import time

#lecture de fichiers
import csv

# pour la fusion
import heapq

os.makedirs("data", exist_ok=True)

In [ ]:
# !rm -f *.csv

## Fonction pour fusionner 2 fichiers triés

Fusion de 2 fichiers csv (clé,valeur) triés par clé croissante.
En cas de clés indentiques dans les 2 fichiers, la valeur associée à la clé provenant du **premier** fichier est retenue.

In [2]:
# fonction auxilliaire pour fusionner 2 fichiers
def fusionner_2_fichiers_csv_tries(file1_path, file2_path, output_file):
    assert(len(set([file1_path, file2_path, output_file])) == 3), 'files should be distincts'
    with open(file1_path, mode='r') as file1, \
        open(file2_path, mode='r') as file2, \
        open(output_file, mode='w', newline='') as outfile:

        reader1 = csv.reader(file1)
        reader2 = csv.reader(file2)
        writer = csv.writer(outfile)

        row1 = next(reader1, None)  # Lire la première ligne du fichier 1
        row2 = next(reader2, None)  # Lire la première ligne du fichier 2

        # Fusion ligne par ligne
        while row1 or row2:
            if row1 and (not row2 or int(row1[0]) < int(row2[0])):
                # La clé du fichier 1 est prioritaire ou le fichier 2 est épuisé
                writer.writerow(row1)
                row1 = next(reader1, None)
            elif row2 and (not row1 or int(row2[0]) < int(row1[0])):
                # La clé du fichier 2 est unique
                writer.writerow(row2)
                row2 = next(reader2, None)
            else:
                # Les clés sont identiques, on garde la valeur du fichier 1
                writer.writerow(row1)
                row1 = next(reader1, None)
                row2 = next(reader2, None)

In [3]:
# Exemple d'utilisation

# création du fichier 1
with open('fichier1.csv', mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([1, 'Z'])
    writer.writerow([3, 'B'])
    writer.writerow([5, 'D'])

# création du fichier 2
with open('fichier2.csv', mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([1, 'A'])
    writer.writerow([2, 'B'])
    writer.writerow([3, 'D'])
    writer.writerow([5, 'C'])

fichier_sortie = 'fichier_fusionne.csv'

###  Question Prep1

Que contient le fichier de sortie apres  l'appel `fusionner_2_fichiers_csv_tries('fichier1.csv', 'fichier2.csv', fichier_sortie)`

 A COMPLETER!

 ...

 ...

 ...

In [4]:
# Vérifier la réponse

fusionner_2_fichiers_csv_tries('fichier1.csv', 'fichier2.csv', fichier_sortie)

print("Le fichier fusionné contient:")
with open(fichier_sortie, 'r') as f:
    for l in csv.reader(f):
      print(l)

Le fichier fusionné contient:
['1', 'Z']
['2', 'B']
['3', 'B']
['5', 'D']


## Fonction pour fusionner N fichiers triés

Fusion de **plusieurs** fichiers csv (clé,valeur) triés par clé croissante

En cas de doublons sur une clé, la valeur contenie dans le fichier ayant la plus **petite position**  (parmi les fichiers ayant la même clé)  est retenue.

In [5]:
def fusionner_plusieurs_fichiers_csv_tries(fichiers_entree, fichier_sortie):
    """
    Fusionne plusieurs fichiers CSV triés contenant des couples (key, value) en un seul fichier trié.
    En cas de doublons, la valeur contenie dans le fichier ayant la plus **petite position**
    (parmi les fichiers ayant la même clé) est retenue.
    """
    file_list = fichiers_entree + [fichier_sortie]
    assert(len(set(file_list)) == len(file_list)), 'files should be distincts'

    # Ouvrir les fichiers d'entrée et le fichier de sortie
    fichiers = [open(fichier, mode='r') for fichier in fichiers_entree]
    sortie = open(fichier_sortie, mode='w', newline='')

    # Initialiser les lecteurs CSV pour les fichiers d'entrée
    lecteurs = [csv.reader(fichier) for fichier in fichiers]
    writer = csv.writer(sortie)

    # Initialiser un heap pour maintenir les données triées par key puis fichier_index
    # Chaque élément du heap est de la forme (key, fichier_index, value)
    heap = []

    # nombre de lignes dans le fichier fusionné
    nb_lines = 0

    # On commence par ajouter la première ligne de chaque fichier dans le heap
    for i, lecteur in enumerate(lecteurs):
        try:
            key, value = next(lecteur)
            while (key == "DELETED"):
              key, value = next(lecteur)

            key=int(key)
            heapq.heappush(heap, (key, i, value))
        except StopIteration:
            # Si un fichier est vide, on l'ignore
            pass

    # Fusionner les données
    dernier_key = None
    while heap:
        # Extraire l'élément ayant la plus petite clé (et le plus petit fichier_index en cas clés identiques)
        key, fichier_index, value = heapq.heappop(heap)

        # Si la clé est différente de la dernière clé écrite, on l'écrit
        if key != dernier_key:
            writer.writerow([key, value])
            nb_lines += 1
            dernier_key = key

        # Ajouter la ligne suivante du même fichier dans le heap
        try:
            next_key, next_value = next(lecteurs[fichier_index])
            while (next_key == "DELETED"):
              next_key, next_value = next(lecteurs[fichier_index])

            next_key=int(next_key)
            heapq.heappush(heap, (next_key, fichier_index, next_value))
        except StopIteration:
            # Si le fichier est terminé, on ne fait rien
            pass

    # Fermer tous les fichiers
    for fichier in fichiers:
        fichier.close()
    sortie.close()

    return nb_lines

In [6]:
# Exemple d'utilisation
with open('fichier1.csv', mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([1, 'Z'])
    writer.writerow([3, 'B'])
    writer.writerow([5, 'D'])

with open('fichier2.csv', mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([1, 'A'])
    writer.writerow([2, 'E'])
    writer.writerow([3, 'D'])
    writer.writerow([5, 'C'])

with open('fichier3.csv', mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([2, 'A'])
    writer.writerow([4, 'A'])
    writer.writerow([5, 'A'])

fichiers_entree = ['fichier1.csv', 'fichier2.csv', 'fichier3.csv']
fichier_sortie = 'fichier_fusionne.csv'


### Question Prep2
Que contient le fichier de sortie apres  l'appel `fusionner_plusieurs_fichiers_csv_tries(fichiers_entree, fichier_sortie)` ?

 A COMPLETER!

 ...

 ...

 ...

In [7]:
# Vérifier la réponse

fusionner_plusieurs_fichiers_csv_tries(fichiers_entree, fichier_sortie)

print("Fichier fusionne contient:")
with open(fichier_sortie, mode='r') as f:
    for l in csv.reader(f):
      print(l)

Fichier fusionne contient:
['1', 'Z']
['2', 'E']
['3', 'B']
['4', 'A']
['5', 'D']


# Classe LSM : Log Structured Merge Storage

La classe LSM définit les principales fonctions du stockage basé sur LSM Tree Store.

In [8]:
import csv
import bisect

class LSM:

  def __init__(self, MAX_IMAGE = 2, MAX_DICT_SIZE = 10):
    self.D = {}
    self.first_image = 0
    self.last_image = 0

    # le nombre max d'images dans L0
    self.MAX_IMAGE = MAX_IMAGE

    # le nombre max de clés dans le dictionnaire en mémoire
    self.MAX_DICT_SIZE = MAX_DICT_SIZE

    # A compléter
    num_L0_images = len([f for f in os.listdir("data") if f.startswith("L0_")])
    if(num_L0_images > 0):
      self.first_image = 1
      self.last_image = num_L0_images

  def stop(self):
    # stocker sur disque le dictionnaire en mémoire
    if len(self.D) > 0:
      self.flush_D()

  # Fonction pour stocker une donnée
  def update(self, k, v):
      self.D[k] = v

      # vider D dans un fichier image si D a atteint sa taille maxi
      if len(self.D) == self.MAX_DICT_SIZE:
        self.flush_D()
        # continuer avec un D vide
        self.D = {}

  # Fonction pour supprimer un clé
  def delete(self, k):
      # à compléter dans l'exerccie 5
      self.D[k] = "DELETED"
      return

  # Fonction pour lire une donnée stockée
  def select(self, k):

    # rechercher la donnée dans D en mémoire
    if k in self.D:
      value = self.D.get(k)
      result = value if value != "DELETED" else None
      if (result is None): print("Rejected at dict")
      return result

    # Sinon, rechercher parmi les images du niveau L0 : de la plus récente à la plus ancienne
    for m in range(self.last_image, self.first_image-1, -1):
      with open(f"data/L0_{m}.csv", "r") as f:
        for k1, v1 in csv.reader(f):
          k1 = int(k1)

          if k1 == k:
              if v1 == "DELETED":
                print("Rejected at L0")
                return None

              # "remonter" (k1,v1) dans D
              self.update(k1, v1)
              return v1

    # recherche dans le niveau L1
    try:
      with open(f"data/L1.csv", "r") as f:
          for k1, v1 in csv.reader(f):
            k1 = int(k1)
            if k1 == k:
              # "remonter" (k1,v1) dans D
              self.update(k1, v1)
              return v1
    except FileNotFoundError:
      pass

    # not found
    print("Rejected at L1")
    return None

  # Fonction pour "décharger" le dictionnaire en mémoire vers le niveau L0
  def flush_D(self):
    if self.first_image == 0:
      self.first_image = 1

    # créer une nouvelle image dans L0
    self.last_image +=1

    with open(f"data/L0_{self.last_image}.csv", "w") as f:
      writer = csv.writer(f)
      # trier les données du dictionnaire
      sorted_entries = sorted(self.D.items())

      for k, v in sorted_entries:
          writer.writerow([k, v])

      print(f"saved {len(self.D)} items in image {self.last_image}")

    # tenter de fusionner L0 avec L1
    nb_image = self.last_image - self.first_image + 1
    if nb_image == self.MAX_IMAGE:
      self.merge_L0_to_L1()

  # Fusionner les données de L0 (sauf la dernière image) vers le niveau L1
  def merge_L0_to_L1(self):

    # sélection des images de L0 à fusionner
    images_L0 = []
    # for m in range(self.last_image, self.first_image-1, -1):
    for m in range(self.last_image-1, self.first_image-1, -1):
      file_path = f"data/L0_{m}.csv"
      if os.path.isfile(file_path):
        images_L0.append(file_path)

    # si il y a déjà des données dans L1
    if os.path.isfile("data/L1.csv"):
      liste_fichiers = images_L0 + ["data/L1.csv"]
      nb_lines = fusionner_plusieurs_fichiers_csv_tries(liste_fichiers, "data/L1_new.csv")
      print('Fusion, nb de lignes dans L1:', nb_lines)

      # remplacer l'ancien fichier L1 par le nouveau
      shutil.move("data/L1_new.csv", "data/L1.csv")
    else:
      # L1 ne contient rien, il suffit de déplacer la premiere image de L0 vers L1
      nb_lines = fusionner_plusieurs_fichiers_csv_tries(images_L0, "data/L1.csv")
      print('Fusion, nb de lignes dans L1:', nb_lines)

    # supprimer les images L0 qui viennent d'être fusionnées
    for file in images_L0:
        os.remove(file)

    # on décale le numéro de la première image
    self.first_image += len(images_L0)
    print("la première image est " + str(self.first_image))

  def select_many(self, k1, k2, n):
    counter = 0
    key_to_query = k1
    result = []

    while(counter < n and key_to_query <= k2):
      v = self.select(key_to_query)
      if v is not None:
        result.append((key_to_query, v))
        counter += 1
      key_to_query += 1

    return result if len(result) > 0 else None

print("class LSM defined")

class LSM defined


#Exercice 1 : Lectures et écritures

## 1) Compléter la classe LSM définie (partiellement) ci-dessus

a) Compéter le `select` pour rechercher une clé dans les niveaux L0 puis L1.

b) Compléter le `flush_D` pour ajouter une image dans L0. Si le nombre maximal d'images dans L0 est atteint, alors déplacer en les fusionnant les images de L0 vers L1.

c) Compléter la fonction `merge_L0_to_L1`

d) Compléter la méthode `init` pour instancier un object LSM en tenant compte des fichiers existants dans L0 et L1.

## 2) Ecritures

On effectue un test d'écriture de plusieurs couples dans le LSM

In [9]:
# on nettoie le dossier ou seront stockés les données
shutil.rmtree("data")
os.makedirs("data", exist_ok=True)

# on initialise le store
store = LSM(MAX_IMAGE = 3, MAX_DICT_SIZE = 10)

nb_update = 100
rng = np.random.default_rng(seed=1)
keys = rng.integers(0, 200, nb_update)
values = rng.integers(0, 1000, nb_update)


t1 = time.time()
for k,v in zip(keys,values):
  store.update(k,v)


print("duree", round( (time.time() - t1) * 1000, 0), "ms")

saved 10 items in image 1
saved 10 items in image 2
saved 10 items in image 3
Fusion, nb de lignes dans L1: 20
la première image est 3
saved 10 items in image 4
saved 10 items in image 5
Fusion, nb de lignes dans L1: 37
la première image est 5
saved 10 items in image 6
saved 10 items in image 7
Fusion, nb de lignes dans L1: 56
la première image est 7
saved 10 items in image 8
saved 10 items in image 9
Fusion, nb de lignes dans L1: 70
la première image est 9
duree 4.0 ms


E1) Modifier le nombre d'écritures `nb_update` dans le test d'écriture et expliquer le déroulement du test  pour les valeurs 9, 11 et 31 de `nb_update`.



on voit qu'on crée une page dans L0 quand taille dict>10 et qu'on flush L0 vers L1 quand taille L0>2

E2) Dans quelle situation, une écriture `update(k,v)`va-t-elle déclencher la création d'une image dans le niveau L0 ?



voir q précédente

E3) Dans quelle situation, une écriture `update(k,v)` va-t-elle déclencher une ecriture dans le niveau L1 ?


voir q précédente

## 3) Lectures

Mesurer la durée moyenne d'une lecture.


In [10]:
# Pour vérifier que la lecture est correcte, on connait à l'avance la valeur la plus récente de chaque k qu'on est censé lire

nb_lectures = 10

# les couples (k,v) avec les v plus récentes (écrites en dernier)
recent = {k:v for k,v in zip(keys,values)}

t1 = time.time()
for i, (k,v) in enumerate(recent.items()):
    v1 = store.select(k)
    print("select", k, "->", v1)
    if v1 is not None:
      v1 = int(v1)
    if v1 != v:
      print(f"Erreur pour la clé {k}! La valeur retrouvée ({v1}) n'est pas celle attendue ({v})")
    if i==nb_lectures:
      break

print("duree", round( (time.time() - t1) * 1000, 0), "ms")
print("duree moyenne pour un select", round( (time.time() - t1) * 1000 / len(recent), 3), "ms")

select 94 -> 69
select 102 -> 105
saved 10 items in image 10
select 151 -> 759
select 190 -> 787
select 6 -> 873
select 28 -> 191
select 164 -> 555
select 189 -> 802
select 49 -> 357
select 62 -> 191
select 173 -> 617
duree 7.0 ms
duree moyenne pour un select 0.092 ms


Le1) Dans quelle situation une lecture `select(k)` va-t-elle déclencher la création d'une image ?

quand le dict est plein : on vient d'ajouter la valeur selectionnée au dict

Le2) Est ce qu'une lecture `select(k)` peut déclencher la modification du niveau L1 ?

oui, si la création d'image dans L0 engendre taille L0 > 2

# Exercice 2 : Fusion de plusieurs images

On voit que les images de L0 sont fusionnées individuellement vers le niveau L1.
Modifier la méthode `merge_L0_to_L1` pour fusionner toutes les images  de L0 vers L1.
Indications :
* Les images L0 et le fichier L1 peuvent contenir des clés identiques. Le cas échéant, seule la valeur associée à **la clé la plus récente** est conservée.
* Dans le niveau L0, les images les plus récentes sont celles qui ont le plus **grand** numéro d'image.
* Parcourir **une seule fois** les données à fusionner (lire une seule fois les  images de L0 et le contenu de L1). Voir la fonction de fusion entre plusieurs fichiers  `fusionner_plusieurs_fichiers_csv_tries` (au début du TP).





In [ ]:
  # COMPLETER

# Exercice 3 : Effet de la fréquence de lecture

Proposer un test de lecture qui mette en évidence le fait que les clés lues fréquemment sont accédées plus rapidement que les clés rares.

On peut spécifier les fréquences par des poids : 50 pour une clé fréquente, 1 pour une clé rare et 10 pour les autres clés.

Dans cet exercice on ne modifie pas la classe LSM. On demande de proposer un scenario de test qui invoque une série d'opérations select.


In [11]:
frequent_key = keys[0]
print('clé frequente', frequent_key)

rare_key = keys[1]
print('clé rare', rare_key)

weights = [50 if k == frequent_key else 1 if k == rare_key else 10 for k in keys]  # Poids pour chaque valeur de clé
print(weights)

# Générer des clés en respectant les pondérations
random_keys = random.choices(keys, weights=weights, k=500)

frequent_counter = 0
temps_frequent = 0
temps_normal = 0
normal_counter = 0

# lecture des clés
for  k in random_keys:
  if k == frequent_key:
    t2 = time.time()
    v2 = store.select(rare_key)
    temps_frequent += (time.time() - t2)
    frequent_counter += 1
  elif k == rare_key:
    continue
    # ne pas lire une clé rare
  else:
    # lire une clé qui n'est ni fréquente ni rare
    t3 = time.time()
    v3 = store.select(rare_key)
    temps_normal += (time.time() - t3)
    normal_counter += 1

# lire la clé rare une seule fois
t1 = time.time()
v1 = store.select(rare_key)
runtime_rare = time.time() - t1

print("duree moyenne pour la lecture d'une clé fréquente", (temps_frequent/frequent_counter), "ms")
print("duree moyenne pour la lecture d'une clé normale", (temps_normal/normal_counter), "ms")
print("duree moyenne pour la lecture d'une clé rare", runtime_rare, "ms")
print("rapport des durée :", (runtime_rare/temps_frequent))

clé frequente 94
clé rare 102
[50, 1, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 1, 10, 10, 10, 10, 10, 10]
duree moyenne pour la lecture d'une clé fréquente 9.681239272608901e-07 ms
duree moyenne pour la lecture d'une clé normale 4.014232127963218e-06 ms
duree moyenne pour la lecture d'une clé rare 0.00011229515075683594 ms
rapport des durée : 3.514925373134328


# Exercice 4 : Amélioration du stockage du niveau L1

On voit que le niveau L1 contient un seul fichier `L1.csv` qui peut grandir indéfiniment.
Cela tend à ralentir la recherche d'une clé "peu fréquente" qui serait dans le niveau L1.

Proposer une classe LSM_v2 pour laquelle le niveau L1 est composé de plusieurs fichiers $L1_1$, $L1_2$, ... tels que
*   chaque fichier contient au maximum MAX_L1_SIZE lignes (=20 par défaut),
*   les clés sont globalement triées dans les fichiers : les clés du fichier $L1_i$ sont toutes inférieures à celles du fichiers $L1_j$ (pour tout $i<j$)

**Indications** :
* la classe LSM_v2 peut hériter de la classe LSM.

* le LSM peut conserver en mémoire des métadonnées décrivant le niveau L1 : par exemple, la plus petite ($kmin_i$) et la plus grande ($kmax_i$) valeur de clé de chaque fichier $L1_i$.





1) Adapter la méthode `init` pour reconstruire les métadonnées décrivant L1
et conserver les valeurs $kmin_i$ et $kmax_i$ de chaque fichier $L1_i$.

2) Adapter la méthode `select(self, k, v)` : si une clé doit être recherchée dans le niveau L1, on demande de lire seulement le fichier $F_i$ pouvant contenir cette clé, tel que $k \in [kmin_i, kmax_i]$

3) Adapter la méthode `merge_L0_to_L1`

Commencer par adapter la fonction de fusion entre plusieurs fichiers  `fusionner_plusieurs_fichiers_csv_tries` de telle sorte que la sortie soit écrite itérativement dans une série de nouveaux fichiers $L1_i$ en commençant par remplir un premier fichier jusqu'à ce qu'il soit plein, puis un deuxième fichier, et ainsi de suite.

La fusion du niveau L0 vers L1 doit fusionner *toutes* les images L0 vers le niveau L1 :
 * les fichiers de $L_1$ dont les clés sont inférieures à toutes les clés de LO n'ont pas besoin d'être lus ni modifiés.
 * tous les fichiers images L0 et les fichiers de L1 à fusionner sont lus ensemble.

* On suppose que la capacité mémoire est limitée : on ne peut pas charger en mémoire plus de MAX_DICT_SIZE clés. Cela implique de ne pas charger toutes les images L0 en mémoire. De même, aucun fichier $L1_i$ ne peut tenir entièrement en mémoire.

* on rappelle qu'il n'y a aucune clé en double dans le niveau L1.


In [12]:
from collections import deque

# A compléer

# def multi_file_iterator(start, end):
#     i = start
#     while i <= end:
#       if os.path.isfile(f"data/L1_{i}.csv"):
#           with open(f"data/L1_{i}.csv", "r") as f:
#               reader = csv.reader(f)
#               for row in reader:

#                   k, v = line.rstrip("\n").split(",", 1)
#                   yield [k, v]
#       i += 1

def fusionner_plusieurs_fichiers_csv_tries_2(fichiers_entree, L1_new_start, indice, new_L1_max_keys):
    """
    Fusionne plusieurs fichiers CSV triés contenant des couples (key, value) en un seul fichier trié.
    En cas de doublons, la valeur contenie dans le fichier ayant la plus **petite position**
    (parmi les fichiers ayant la même clé) est retenue.
    """

    file_list = fichiers_entree
    assert(len(set(file_list)) == len(file_list)), 'files should be distincts'

    # Ouvrir les fichiers d'entrée et le fichier de sortie
    fichiers = [open(fichier, mode='r') for fichier in fichiers_entree]
    sortie = open(f"data/L1_{L1_new_start + indice}.csv", mode='w', newline='')

    # Initialiser les lecteurs CSV pour les fichiers d'entrée
    lecteurs = [csv.reader(fichier) for fichier in fichiers] # + [multi_file_iterator(L1_new_start, L1_current_end)]
    writer = csv.writer(sortie)

    # Initialiser un heap pour maintenir les données triées par key puis fichier_index
    # Chaque élément du heap est de la forme (key, fichier_index, value)
    heap = []

    # nombre de lignes dans le fichier fusionné
    nb_lines = 0

    # On commence par ajouter la première ligne de chaque fichier dans le heap
    for i, lecteur in enumerate(lecteurs):
        try:
            key, value = next(lecteur)
            while (key == "DELETED"):
              key, value = next(lecteur)

            key=int(key)
            heapq.heappush(heap, (key, i, value))
        except StopIteration:
            # Si un fichier est vide, on l'ignore
            pass

    # Fusionner les données
    dernier_key = None
    while heap:
        # Extraire l'élément ayant la plus petite clé (et le plus petit fichier_index en cas clés identiques)
        key, fichier_index, value = heapq.heappop(heap)

        # Si la clé est différente de la dernière clé écrite, on l'écrit
        if key != dernier_key:
            writer.writerow([key, value])
            nb_lines += 1
            dernier_key = key

            # 20 est la taille max de L1, je prends la liberté de ne pas le passer en paramètre car on en a déjà beaucoup
            if nb_lines % 20 == 0:
              new_L1_max_keys.append(key)
              sortie.close()

              indice += 1
              sortie = open(f"data/L1_{L1_new_start + indice}.csv", mode='w', newline='')
              writer = csv.writer(sortie)

        # Ajouter la ligne suivante du même fichier dans le heap
        try:
            next_key, next_value = next(lecteurs[fichier_index])
            while (next_key == "DELETED"):
              next_key, next_value = next(lecteurs[fichier_index])

            next_key=int(next_key)
            heapq.heappush(heap, (next_key, fichier_index, next_value))
        except StopIteration:
            # Si le fichier est terminé, on ne fait rien
            pass

    # Fermer tous les fichiers
    for fichier in fichiers:
        fichier.close()

    if (nb_lines % 20 != 0):
      new_L1_max_keys.append(key)
    sortie.close()

    return nb_lines

class LSM_v2(LSM):
  def __init__(self, MAX_IMAGE = 2, MAX_DICT_SIZE = 10):
    self.D = {}
    self.first_image = 0
    self.last_image = 0
    self.min_L0_key = None

    # le nombre max d'images dans L0
    self.MAX_IMAGE = MAX_IMAGE

    # le nombre max de clés dans le dictionnaire en mémoire
    self.MAX_DICT_SIZE = MAX_DICT_SIZE

    # A compléter
    num_L0_images = len([f for f in os.listdir("data") if f.startswith("L0_")])
    if(num_L0_images > 0):
      self.first_image = 1
      self.last_image = num_L0_images

      for m in range(self.first_image, self.last_image+1):
        with open(f"data/L0_{m}.csv", "r") as f:
          first_key, first_value = next(csv.reader(f))
          first_key = int(first_key)
          if self.min_L0_key is None or first_key < self.min_L0_key:
            self.min_L0_key = first_key

    self.MAX_L1_SIZE = 20
    self.first_image_L1 = 0
    self.last_image_L1 = 0
    self.L1_max_keys = []

    num_L1_images = len([f for f in os.listdir("data") if f.startswith("L1_")])
    if(num_L1_images > 0):
      self.first_image_L1 = 1
      self.last_image_L1 = num_L1_images

      for m in range(self.first_image_L1, self.last_image_L1+1):
        with open(f"data/L1_{m}.csv", "r") as f:
          last_line = deque(f, maxlen=1)
          if last_line:
            self.L1_max_keys.append(int(last_line[0].rstrip("\n").split(",", 1)[0]))
          else:
            print("Error: L1 file is empty")

  # Fonction pour lire une donnée stockée
  def select(self, k):

    # rechercher la donnée dans D en mémoire
    if k in self.D:
      value = self.D.get(k)
      result = value if value != "DELETED" else None
      if (result is None): print("Rejected at dict")
      return result

    # Sinon, rechercher parmi les images du niveau L0 : de la plus récente à la plus ancienne
    for m in range(self.last_image, self.first_image-1, -1):
      with open(f"data/L0_{m}.csv", "r") as f:
        for k1, v1 in csv.reader(f):
          k1 = int(k1)

          if k1 == k:
              if v1 == "DELETED":
                print("Rejected at L0")
                return None

              # "remonter" (k1,v1) dans D
              self.update(k1, v1)
              return v1

    # Sinon, chercher dans L1
    indice = bisect.bisect_left(self.L1_max_keys, k)
    indice = min(self.first_image_L1 + indice, self.last_image_L1)
    with open(f"data/L1_{indice}.csv", "r") as f:
      for k1, v1 in csv.reader(f):
        k1 = int(k1)
        if k1 == k:
          # "remonter" (k1,v1) dans D
          self.update(k1, v1)
          return v1

    # L1_key = self.first_image_L1
    # starts = [v[0] for _, v in self.L1_values.items()]
    # i = bisect.bisect_right(starts, k) - 1
    # if i >= 0 and self.intervals[i][0] <= k < self.intervals[i][1]:
    #   L1_key = self.first_image_L1 + self.intervals[i] + 1
    # print(L1_key)
    # try:
    #   with open(f"data/L1_{L1_key}.csv", "r") as f:
    #       for k1, v1 in csv.reader(f):
    #         k1 = int(k1)
    #         if k1 == k:
    #           # "remonter" (k1,v1) dans D
    #           self.update(k1, v1)
    #           return v1
    # except FileNotFoundError:
    #   pass

    print("Rejected at L1")
    return None

  # Fonction pour "décharger" le dictionnaire en mémoire vers le niveau L0
  def flush_D(self):
    if self.first_image == 0:
      self.first_image = 1

    # créer une nouvelle image dans L0
    self.last_image +=1

    with open(f"data/L0_{self.last_image}.csv", "w") as f:
      writer = csv.writer(f)
      # trier les données du dictionnaire
      sorted_entries = sorted(self.D.items())

      min_entry = int(sorted_entries[0][0])
      if self.min_L0_key is None or min_entry < self.min_L0_key:
          self.min_L0_key = min_entry

      for k, v in sorted_entries:
          writer.writerow([k, v])

      print(f"saved {len(self.D)} items in image {self.last_image}")

    # tenter de fusionner L0 avec L1
    nb_image = self.last_image - self.first_image + 1
    if nb_image == self.MAX_IMAGE:
      self.merge_L0_to_L1()

  # Fusionner les données de L0 (sauf la dernière image) vers le niveau L1
  def merge_L0_to_L1(self):
    # sélection des images de L0 à fusionner
    images_L0 = []
    # for m in range(self.last_image, self.first_image-1, -1):
    for m in range(self.last_image-1, self.first_image-1, -1):
      file_path = f"data/L0_{m}.csv"
      if os.path.isfile(file_path):
        images_L0.append(file_path)

    new_L1_max_keys = []
    new_first_image_L1 = self.last_image_L1 + 1
    indice = 0

    if (self.min_L0_key is not None and self.first_image_L1 > 0):
      for m in range(self.first_image_L1, self.last_image_L1 + 1):
        with open(f"data/L1_{m}.csv", "r", newline="") as f:
          last_key = None
          last_line = deque(f, maxlen=1)
          if last_line:
            last_key = int(last_line[0].rstrip("\n").split(",", 1)[0])
          else:
            print("Error: L1 file is empty")

          if (last_key is not None and last_key < self.min_L0_key):
            # écrire le fichier dans le nouveau L1
            os.rename(f"data/L1_{m}.csv", f"data/L1_{new_first_image_L1 + indice}.csv")

            new_L1_max_keys.append(last_key)
            indice += 1
          else:
            break

    files_to_merge = images_L0 + [f"data/L1_{i}.csv" for i in range(self.first_image_L1 + indice, self.last_image_L1 + 1) if os.path.isfile(f"data/L1_{i}.csv")]
    nb_lines = fusionner_plusieurs_fichiers_csv_tries_2(files_to_merge, new_first_image_L1, indice, new_L1_max_keys)
    print('Fusion, nb de lignes dans L1:', nb_lines)

    # supprimer les images L0 et L1 qui viennent d'être fusionnées
    for file in files_to_merge:
        os.remove(file)

    # Update des valeurs de L0
    self.first_image += len(images_L0)
    print("la première image est " + str(self.first_image))
    # On reset la valeur de la clé minimum de L0
    for m in range(self.first_image, self.last_image+1):
        with open(f"data/L0_{m}.csv", "r") as f:
          first_key, first_value = next(csv.reader(f))
          first_key = int(first_key)
          if self.min_L0_key is None or first_key < self.min_L0_key:
            self.min_L0_key = first_key

    # Update des valeurs de L1
    self.L1_max_keys = new_L1_max_keys
    self.first_image_L1 = new_first_image_L1
    self.last_image_L1 = new_first_image_L1 + indice + ((nb_lines // self.MAX_L1_SIZE) - 1) + (1 if nb_lines % self.MAX_L1_SIZE != 0 else 0)
    print("L1 va de " + str(self.first_image_L1) + " à " + str(self.last_image_L1))

# Ecriture
# on nettoie le dossier ou seront stockés les données
shutil.rmtree("data")
os.makedirs("data", exist_ok=True)

# on initialise le store
store_2 = LSM_v2(MAX_IMAGE = 3, MAX_DICT_SIZE = 10)

nbs_2 = 100
rng_2 = np.random.default_rng(seed=1)
keys_2 = rng_2.integers(0, 200, nb_update)
values_2 = rng_2.integers(0, 1000, nb_update)

t1 = time.time()
for k,v in zip(keys_2,values_2):
  store_2.update(k,v)

print("duree", round( (time.time() - t1) * 1000, 0), "ms")

# Lecture
recent_2 = {k:v for k,v in zip(keys_2,values_2)}
t1 = time.time()
for i, (k,v) in enumerate(recent_2.items()):
    v2 = store_2.select(k)
    print("select", k, "->", v2)
    if v2 is not None:
      v2 = int(v2)
    if v2 != v:
      print(f"Erreur pour la clé {k}! La valeur retrouvée ({v2}) n'est pas celle attendue ({v})")
    if i==nbs_2:
      break

print("duree", round( (time.time() - t1) * 1000, 0), "ms")
print("duree moyenne pour un select", round( (time.time() - t1) * 1000 / len(recent_2), 3), "ms")

saved 10 items in image 1
saved 10 items in image 2
saved 10 items in image 3
Fusion, nb de lignes dans L1: 20
la première image est 3
L1 va de 1 à 1
saved 10 items in image 4
saved 10 items in image 5
Fusion, nb de lignes dans L1: 37
la première image est 5
L1 va de 2 à 3
saved 10 items in image 6
saved 10 items in image 7
Fusion, nb de lignes dans L1: 56
la première image est 7
L1 va de 4 à 6
saved 10 items in image 8
saved 10 items in image 9
Fusion, nb de lignes dans L1: 70
la première image est 9
L1 va de 7 à 10
duree 4.0 ms
select 94 -> 69
select 102 -> 105
saved 10 items in image 10
select 151 -> 759
select 190 -> 787
select 6 -> 873
select 28 -> 191
select 164 -> 555
select 189 -> 802
select 49 -> 357
select 62 -> 191
select 173 -> 617
select 84 -> 679
saved 10 items in image 11
Fusion, nb de lignes dans L1: 80
la première image est 11
L1 va de 11 à 14
select 54 -> 219
select 165 -> 855
select 51 -> 667
select 81 -> 861
select 128 -> 691
select 109 -> 24
select 17 -> 310
select

# Exercice 5 : suppression de clé  

Compléter la classe LSM pour gérer la suppression d'une clé.

Implémenter la méthode `delete(self, k)` afin qu'elle soit la plus rapide possible et déclenche un minimum d'écritures.

Modifier en conséquence la méthode `select` pour ne pas retourner une clé sachant qu'elle a été supprimée.

Modifier aussi la méthode `merge_L0_to_L1` pour que L1 ne contienne aucune clé supprimée.


Implanter aussi la suppression d'un clé dans la classe LSM_v2

In [ ]:
# compléter

my_keys = list(set(keys_2[30:45]))

for  k3 in my_keys:
  v3 = store_2.select(k3)
  print("select", k3, "->", v3)

print("----------")
key_to_delete = my_keys[1]
store_2.delete(key_to_delete)
print("DELETING:", key_to_delete)
print("----------")

for  k4 in my_keys:
  v4 = store_2.select(k4)
  print("select", k4, "->", v4)


select 195 -> 470
select 100 -> 976
saved 10 items in image 18
select 90 -> 281
select 3 -> 894
select 56 -> 482
select 40 -> 805
select 99 -> 345
select 76 -> 692
select 12 -> 316
select 80 -> 639
select 180 -> 844
select 52 -> 5
saved 10 items in image 19
Fusion, nb de lignes dans L1: 80
la première image est 19
L1 va de 27 à 30
select 150 -> 633
select 24 -> 998
select 26 -> 215
----------
DELETING: 100
----------
select 195 -> 470
Rejected at dict
select 100 -> None
select 90 -> 281
select 3 -> 894
select 56 -> 482
select 40 -> 805
select 99 -> 345
saved 10 items in image 20
select 76 -> 692
select 12 -> 316
select 80 -> 639
select 180 -> 844
select 52 -> 5
select 150 -> 633
select 24 -> 998
select 26 -> 215


#Exercice 6 : sélection multiple

Définir une méthode `select_many(self, k1, k2, n)` pour lire les $n$ entrées (k,v) les plus petites comprises dans l'intervalle fermé $[k_1, k_2]$.

Si l'intervalle contient m clés et m<n, alors la fonction retourne seulement les m clés.



In [13]:
# compléter

print(store_2.select_many(10, 100, 4))
print("----------")
print(store_2.select_many(10, 12, 50))

Rejected at L1
Rejected at L1
Rejected at L1
Rejected at L1
Rejected at L1
saved 10 items in image 18
[(10, '767'), (12, '316'), (17, '310'), (18, '890')]
----------
Rejected at L1
[(10, '767'), (12, '316')]
